Import the libraries.

In [62]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline

Read the data.

In [63]:
df = pd.read_csv('Datasets/titanic.csv', usecols=['Age', 'Fare', 'Survived'])
df.sample(5)

,Survived,Age,Fare
385,0,18.0,73.5000
144,0,18.0,11.5000
155,0,51.0,61.3792
606,0,30.0,7.8958
667,0,NaN,7.7750


Train test split.

In [64]:
x = df.drop('Survived', axis=1)
y = df['Survived']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train.head()

,Age,Fare
331,45.5,28.5000
733,23.0,13.0000
382,32.0,7.9250
704,26.0,7.8542
813,6.0,31.2750


Simple imputation on train set.

In [65]:
si = SimpleImputer()
x_train_trf = si.fit_transform(x_train)
x_test_trf = si.transform(x_test)
x_train_trf[:5]


array([[45.5   , 28.5   ],
       [23.    , 13.    ],
       [32.    ,  7.925 ],
       [26.    ,  7.8542],
       [ 6.    , 31.275 ]])

Train the model and find accuracy score.

In [66]:
clf = LogisticRegression()
clf.fit(x_train_trf, y_train)

y_pred = clf.predict(x_test_trf)
accuracy_score(y_test, y_pred)
#  Above code can be done by this also but it is not recommended as it does not give us the flexibility to check the performance of the model on the training data.
# clf.score(x_test_trf, y_test)

0.6480446927374302

Missing Indicator - Create a new column where null/missing value are represented by 0.

In [67]:
mi = MissingIndicator()
mi.fit(x_train)

,missing_values,nan
,features,'missing-only'
,sparse,'auto'
,error_on_new,True


In [68]:
mi.features_

array([0])

Transform the data, i.e, create a new column.

In [69]:
x_train_mi = mi.transform(x_train)
x_test_mi = mi.transform(x_test)
x_train_mi[:5], x_test_mi[:5]

(array([[False],
        [False],
        [False],
        [False],
        [False]]),
 array([[ True],
        [False],
        [False],
        [False],
        [False]]))

Add new column into dataset.

In [70]:
x_train['Age_NA'] = x_train_mi
x_test['Age_NA'] = x_test_mi
x_train.head(), x_test.head()

(      Age     Fare  Age_NA
 331  45.5  28.5000   False
 733  23.0  13.0000   False
 382  32.0   7.9250   False
 704  26.0   7.8542   False
 813   6.0  31.2750   False,
       Age     Fare  Age_NA
 709   NaN  15.2458    True
 439  31.0  10.5000   False
 840  20.0   7.9250   False
 720   6.0  33.0000   False
 39   14.0  11.2417   False)

Transform the data.

In [71]:
x_train_trf2 = si.fit_transform(x_train)
x_test_trf2 = si.transform(x_test)
x_train_trf2[:5]

array([[45.5   , 28.5   ,  0.    ],
       [23.    , 13.    ,  0.    ],
       [32.    ,  7.925 ,  0.    ],
       [26.    ,  7.8542,  0.    ],
       [ 6.    , 31.275 ,  0.    ]])

Train the model and find the accuracy score.

In [72]:
clf.fit(x_train_trf2, y_train)
y_pred2 = clf.predict(x_test_trf2)
accuracy_score(y_test, y_pred2)

0.6368715083798883

Train test split.

In [73]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

Make a pipeline for different preprocessing and find the accuracy score.

In [74]:
pipe = Pipeline([
    ("imputer", SimpleImputer(add_indicator=True)),
    ("model", LogisticRegression(max_iter=1000))
])

pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)

accuracy_score(y_test, y_pred)

0.6368715083798883

Selection of best parameter by following.

In [75]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler 

In [76]:
dfm1 = pd.read_csv('Datasets/titanic.csv')
dfm1 = dfm1.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=False)
dfm1.sample(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
313,0,3,male,28.0,0,0,7.8958,S
127,1,3,male,24.0,0,0,7.1417,S
539,1,1,female,22.0,0,2,49.5000,C
518,1,2,female,36.0,1,0,26.0000,S
764,0,3,male,16.0,0,0,7.7750,S


Train test split.

In [77]:
x3 = dfm1.drop('Survived', axis=1)
y3 = dfm1['Survived']
x_train3, x_test3, y_train3, y_test3 = train_test_split(x3, y3, test_size=0.2, random_state=42)
x_train3.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


Column transformer for the differert preprocessing of different columns. And Pipelines to chain multiple steps together.

In [78]:
numeric_features = ['Age', 'Fare']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categotric_features = ['Sex', 'Embarked']
categoric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categoric_transformer, categotric_features)
])

clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])
clf # To see the pipeline as a diagram. Same can be done by:- 
# from sklearn import set_config
# set_config(display='diagram')
# clf

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Tuning model by trying and running all different combinations (stepname__substep__parameter).

In [79]:
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'preprocessor__cat__imputer__strategy': ['most_frequent', 'constant'],
    'classifier__C': [0.1, 1, 10, 100]
}
gridsearch = GridSearchCV(clf, param_grid, cv=5)
gridsearch.fit(x_train3, y_train3)
print("Best Hyperparameters:", gridsearch.best_params_)


Best Hyperparameters: {'classifier__C': 0.1, 'preprocessor__cat__imputer__strategy': 'most_frequent', 'preprocessor__num__imputer__strategy': 'mean'}


Print the best parameter and its score.

In [80]:
print(f"Internal CV score: {gridsearch.best_score_:.3f}")

Internal CV score: 0.784


Show the results as a dataframe.

In [81]:
cv_results = pd.DataFrame(gridsearch.cv_results_)
cv_results.sort_values('mean_test_score', ascending=False).head()
cv_results[['param_classifier__C','param_preprocessor__cat__imputer__strategy','param_preprocessor__num__imputer__strategy','mean_test_score']]

,param_classifier__C,param_preprocessor__cat__imputer__strategy,param_preprocessor__num__imputer__strategy,mean_test_score
0,0.1,most_frequent,mean,0.78367
1,0.1,most_frequent,median,0.78367
2,0.1,constant,mean,0.78367
3,0.1,constant,median,0.78367
4,1.0,most_frequent,mean,0.78367
5,1.0,most_frequent,median,0.78367
6,1.0,constant,mean,0.78367
7,1.0,constant,median,0.78367
8,10.0,most_frequent,mean,0.78367
9,10.0,most_frequent,median,0.78367
